# Phase 2 — Self-Consistency and Confidence Estimation

This notebook implements self-consistency decoding for improving the reliability of Large Language Model (LLM)-based question answering.

Instead of relying on a single generated answer, the model produces multiple stochastic generations for the same question.

The final answer is selected using majority agreement among generated responses.

This phase introduces:
- repeated stochastic sampling,
- answer aggregation,
- confidence estimation,
- and confidence-based abstention.

The goal is to improve robustness and reduce unreliable responses.

In [7]:
# ==========================================================
# IMPORT REQUIRED LIBRARIES
# ==========================================================

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

from collections import Counter

In [9]:
# ==========================================================
# LOAD FLAN-T5 MODEL
# ==========================================================

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
# ==========================================================
# BASELINE QA GENERATION FUNCTION
# ==========================================================

def qa_model(question):

    inputs = tokenizer(
        question,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_length=64,
        do_sample=True,
        temperature=0.8
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [18]:
# ==========================================================
# RESPONSE VARIABILITY EXPERIMENT
# ==========================================================

question = "What is Numerical Linear Algebra?"

for i in range(10):

    answer = qa_model(question)

    print(f"Sample {i+1}: {answer}")

Sample 1: linear algebra
Sample 2: nonlinear geometrical formula
Sample 3: simple linear algebra
Sample 4: a graph of multiplication
Sample 5: nonlinear algebra
Sample 6: formula
Sample 7: algebra
Sample 8: linear algebra
Sample 9: formula
Sample 10: arithmetic


# Observations on Generation Variability

The generated answers may vary considerably across stochastic generations.

This variability reveals that:
- LLM outputs are not fully deterministic,
- technical questions may produce unstable answers,
- and uncertainty estimation is necessary.

To address this issue, we introduce self-consistency aggregation.

In [21]:
# ==========================================================
# SELF-CONSISTENCY FUNCTION
# ==========================================================

def self_consistency(question, n_samples=10):

    answers = []

    # ------------------------------------------------------
    # GENERATE MULTIPLE ANSWERS
    # ------------------------------------------------------

    for _ in range(n_samples):

        answer = qa_model(question)

        answers.append(answer)

    # ------------------------------------------------------
    # NORMALIZE ANSWERS
    # ------------------------------------------------------

    normalized_answers = [
        a.strip().lower().replace(".", "")
        for a in answers
    ]

    # ------------------------------------------------------
    # COUNT ANSWER FREQUENCIES
    # ------------------------------------------------------

    counts = Counter(normalized_answers)

    final_answer, max_count = counts.most_common(1)[0]

    confidence = max_count / n_samples

    return {
        "question": question,
        "answers": answers,
        "final_answer": final_answer,
        "confidence": confidence
    }

In [23]:
# ==========================================================
# TEST SELF-CONSISTENCY
# ==========================================================

result = self_consistency(
    "What is Numerical Linear Algebra?",
    n_samples=10
)

result

{'question': 'What is Numerical Linear Algebra?',
 'answers': ['mathematical linear algebra',
  'arithmetic',
  'multiplication',
  'algebra',
  'mathematics',
  'linear algebra',
  'algorithm for arithmetic equations',
  'linear algebra',
  'algebras',
  'algebra'],
 'final_answer': 'algebra',
 'confidence': 0.2}

# Self-Consistency Interpretation

Self-consistency aggregates multiple stochastic generations and selects the most frequent answer.

The confidence score is estimated as:

\[
\text{Confidence} =
\frac{
\text{Frequency of Most Common Answer}
}{
\text{Number of Samples}
}
\]

Higher confidence indicates stronger agreement among generated answers.

This approach improves robustness and prepares the system for later reliability-oriented decision mechanisms.

In [26]:
# ==========================================================
# CONFIDENCE-BASED DECISION FUNCTION
# ==========================================================

def reliable_qa(question, threshold=0.4):

    result = self_consistency(question)

    raw_answer = result["final_answer"]

    confidence = result["confidence"]

    # ------------------------------------------------------
    # MODEL ABSTENTION CHECK
    # ------------------------------------------------------

    is_model_abstain = (
        raw_answer.strip().lower() == "i don't know"
    )

    # ------------------------------------------------------
    # FINAL DECISION
    # ------------------------------------------------------

    if confidence < threshold or is_model_abstain:

        decision = "I don't know"

    else:

        decision = raw_answer

    return {
        "question": question,
        "raw_answer": raw_answer,
        "confidence": confidence,
        "final_decision": decision,
        "model_abstained": is_model_abstain
    }

In [28]:
# ==========================================================
# TEST RELIABLE QA SYSTEM
# ==========================================================

questions = [
    "What is machine learning?",
    "What is Numerical Linear Algebra?",
    "Who is Albert Einstein?"
]

for q in questions:

    result = reliable_qa(q, threshold=0.4)

    print("=" * 50)
    print("Question:", q)
    print("Raw Answer:", result["raw_answer"])
    print("Confidence:", result["confidence"])
    print("Final Decision:", result["final_decision"])

Question: What is machine learning?
Raw Answer: machine learning is a computer science science experiment
Confidence: 0.1
Final Decision: I don't know
Question: What is Numerical Linear Algebra?
Raw Answer: linear algebra
Confidence: 0.3
Final Decision: I don't know
Question: Who is Albert Einstein?
Raw Answer: physicist
Confidence: 0.7
Final Decision: physicist


# Phase 2 Summary

This phase introduced self-consistency and confidence-based abstention for improving QA reliability.

Main contributions of this phase include:
- repeated stochastic generation,
- answer aggregation,
- confidence estimation,
- and selective abstention.

Experimental observations suggest that:
- self-consistency improves answer stability,
- confidence scores provide useful uncertainty signals,
- and abstention mechanisms reduce unreliable outputs.

These developments prepare the system for later calibration and retrieval-based improvements.